In [1]:
import os
from functools import partial
import gc
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import jax.numpy as jnp

from feature_functions import (
    get_artclip_score_model,
)
from torchvision import transforms
import torch
from transformers import AutoModel, AutoImageProcessor

### Load in Dataset

In [2]:
def load_images_from_folder(folder, max_images=None):
    supported_extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG"]
    image_paths = []
    for ext in supported_extensions:
        image_paths.extend(glob.glob(os.path.join(folder, ext)))

    if max_images:
        image_paths = image_paths[:max_images]

    print(f"Found {len(image_paths)} image files.")

    images = []
    for path in image_paths:
        try:
            with Image.open(path) as img:
                img = img.convert("RGB")  # Important: force RGB
                images.append(img.copy())  # .copy() keeps image after context closes
        except Exception as e:
            print(f"Error loading {path}: {e}")

    return images

In [3]:
descriptor_names = [
    # "artclip_score",
    # "theme_and_logic",
    # "creativity",
    # "layout_and_composition",
    # "space_and_perspective",
    # # "sense_of_order",
    # "light_and_shadow",
    # "color",
    # "details_and_texture",
    # "overall",
    # "mood",
    "dino"
]

In [4]:
def compute_and_plot_descriptors(images, descriptor_names, folder):
    # Pre-load DINOv2 model and projection if needed
    device = "cuda" if torch.cuda.is_available() else "cpu"

    dino_model = None
    dino_processor = None
    projection_matrix = None

    for name in descriptor_names:
        print(f"Processing descriptor: {name}")

        if name == "dino":
            # Lazy load once
            if dino_model is None:
                from transformers import AutoImageProcessor, AutoModel
                dino_processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
                dino_model = AutoModel.from_pretrained("facebook/dinov2-base").eval().to(device)
                np.random.seed(42)
                projection_matrix = np.random.randn(2, 768)

            def descriptor_fn(img):
                inputs = dino_processor(images=img, return_tensors="pt").to(device)
                with torch.no_grad():
                    output = dino_model(**inputs)
                dino_embedding = output.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()  # shape: (768,)
                features = projection_matrix @ dino_embedding  # shape: (2,)
                return features

        else:
            descriptor_fn = get_artclip_score_model(name)

        # Compute descriptor values
        values = []
        for img in images:
            try:
                score = descriptor_fn(img)
                values.append(score)
            except Exception as e:
                print(f"Failed to score image with {name}: {e}")

        values = np.array(values)

        os.makedirs(os.path.join(folder, "histograms"), exist_ok=True)

        if name == "dino":
            # 2D scatter plot
            plt.figure()
            plt.scatter(values[:, 0], values[:, 1], alpha=0.5, s=10)
            plt.title("DINOv2 2D Projection")
            plt.xlabel("DINO Feature 1")
            plt.ylabel("DINO Feature 2")
            plt.grid(True)
            plt.tight_layout()

            save_path = os.path.join(folder, "histograms", "dino_scatter.png")
        else:
            # 1D histogram
            plt.figure()
            plt.hist(values, bins=30, edgecolor='black')
            plt.title(f"Distribution of {name}")
            plt.xlabel(name)
            plt.ylabel("Frequency")
            plt.grid(True)
            plt.tight_layout()

            save_path = os.path.join(folder, "histograms", f"{name}_histogram.png")

        print("Saving plot to:", save_path)
        plt.savefig(save_path)
        plt.close()

        del descriptor_fn, values
        gc.collect()

In [6]:
folder = "cat_images_dataset"  # Change to your dataset folder
max_images = 15000  # adjust to fit memory

print("Loading images...")
images_np = load_images_from_folder(folder, max_images=max_images)
print(f"Loaded {len(images_np)} images.")

compute_and_plot_descriptors(images_np, descriptor_names, folder)
print("All descriptor histograms saved.")

Loading images...
Found 18 image files.
Loaded 18 images.
Processing descriptor: dino
Saving plot to: cat_images_dataset/histograms/dino_scatter.png
All descriptor histograms saved.


In [29]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import glob
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

def load_pil_images_from_folder(folder, max_images=None):
    extensions = ["*.jpg", "*.jpeg", "*.png"]
    paths = []
    for ext in extensions:
        paths.extend(glob.glob(os.path.join(folder, ext)))

    if max_images:
        paths = paths[:max_images]

    images = []
    for path in paths:
        try:
            with Image.open(path) as img:
                images.append((img.convert("RGB").copy(), path))
        except Exception as e:
            print(f"Error loading {path}: {e}")
    return images

def compute_dino_2d_descriptors(images, processor, model, projection_matrix, device):
    descriptors = []
    for img, _ in images:
        try:
            inputs = processor(images=img, return_tensors="pt").to(device)
            with torch.no_grad():
                output = model(**inputs)
            embedding = output.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
            projected = projection_matrix @ embedding  # (2,)
            descriptors.append(projected)
        except Exception as e:
            print(f"Failed to process image: {e}")
            descriptors.append(np.array([np.nan, np.nan]))  # Placeholder for failed
    return np.array(descriptors)

def compare_datasets_with_dino_scatter_images(dataset1_path, dataset2_path, output_path="dino_scatter_thumbnails.png", max_images=200):
    # Set up DINOv2
    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
    model = AutoModel.from_pretrained("facebook/dinov2-base").eval().to(device)

    # Fixed 2D projection
    np.random.seed(42)
    projection_matrix = np.random.randn(2, 768)

    # Load images
    images1 = load_pil_images_from_folder(dataset1_path, max_images)
    images2 = load_pil_images_from_folder(dataset2_path, max_images)

    # Compute 2D projections
    descriptors1 = compute_dino_2d_descriptors(images1, processor, model, projection_matrix, device)
    descriptors2 = compute_dino_2d_descriptors(images2, processor, model, projection_matrix, device)

    # Combine for plotting
    all_points = np.concatenate([descriptors1, descriptors2], axis=0)
    all_images = [img for img, _ in images1] + [img for img, _ in images2]
    dataset_labels = [os.path.basename(dataset1_path)] * len(images1) + [os.path.basename(dataset2_path)] * len(images2)

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['red' if lbl == os.path.basename(dataset1_path) else 'blue' for lbl in dataset_labels]

    ax.scatter(all_points[:, 0], all_points[:, 1], s=5, c=colors, alpha=0.3)

    # Add thumbnails (downsampled)
    for (x, y), img in zip(all_points, all_images):
        if np.isnan(x) or np.isnan(y):
            continue
        try:
            thumbnail = img.resize((256, 256))
            imagebox = OffsetImage(thumbnail, zoom=0.15)
            ab = AnnotationBbox(imagebox, (x, y), frameon=False, pad=0.1)
            ax.add_artist(ab)
        except Exception as e:
            print(f"Error adding thumbnail: {e}")

    ax.set_title("DINOv2 2D Projection with Thumbnails")
    ax.set_xlabel("DINO Feature 1")
    ax.set_ylabel("DINO Feature 2")
    legend_handles = [plt.Line2D([], [], color='red', label=os.path.basename(dataset1_path), marker='o', linestyle=''),
                      plt.Line2D([], [], color='blue', label=os.path.basename(dataset2_path), marker='o', linestyle='')]
    ax.legend(handles=legend_handles)
    ax.grid(True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()
    print(f"Saved scatter plot with thumbnails to: {output_path}")

In [31]:
compare_datasets_with_dino_scatter_images(
    dataset1_path="joel_simon_dataset",
    dataset2_path="joel_simon_dataset",
    output_path="scatter_joel_simon.png",
    max_images=500
)

Saved scatter plot with thumbnails to: scatter_joel_simon.png
